In [1]:
import pandas as pd
from src.pipelines.BERT_pipeline import BERTPipeline
from src.pipelines.SBERT_pipeline import SBERTPipeline
import logging
import torch
import os

In [2]:
df = pd.read_csv("data/aes_dataset_5k_clean.csv")
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [3]:
# Check if the first file exists
df_result = None
if os.path.exists("experiments/results/results_sbert.csv"):
    df_result = pd.read_csv("experiments/results/results_sbert.csv")
    print(df_result['config_id'].iloc[-1])
else:
    print("File 'results_sbert.csv' does not exist.")

1


In [ ]:
batch_sizes = [4]
learning_rates = [1e-5]
warm_ups = [0.0]
idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
ROOT_DIR = os.getcwd()

In [ ]:
for batch_size in batch_sizes:
    for lr in learning_rates:
        for warm_up in warm_ups:
            results = []
            results_epoch = []
            df_result1 = None
            # Check if the second file exists
            if os.path.exists("experiments/results/results_epoch_sbert.csv"):
                df_result1 = pd.read_csv("experiments/results/results_epoch_sbert.csv")
                print(max(df_result1['valid_pearson']))
            else:
                print("File 'results_epoch_sbert.csv' does not exist.")

            # set up hyperparamter
            config = {
                "df": df,
                # "model_name": "indobenchmark/indobert-lite-base-p2",
                "model_name": "all-MiniLM-L6-v2",
                "batch_size": batch_size,
                "learning_rate": lr,
                "epochs": 100,
                "config_id": idx,
                "best_valid_pearson": max(df_result1['valid_pearson']) if df_result1 is not None and not df_result1.empty else float("-inf"),
                "warmup_ratio": warm_up,
            }

            logging.info(
                f"Running configuration: config_id={idx}, model_name={config['model_name']}"
                f", batch_size={batch_size}, epochs={100}, learning_rate={lr}"
            )
            
            print(
                f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}"
                f", batch_size={batch_size}, epochs={100}, learning_rate={lr}"
            )
            
            try:
                pipeline = SBERTPipeline(config, results, results_epoch)
                pipeline.training()

                # Save results
                # Dapatkan root project
                results_path = os.path.join(ROOT_DIR, "experiments/results/results_sbert.csv")
                results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch_sbert.csv")
                BERTPipeline.save_csv(results, results_path)
                BERTPipeline.save_csv(results_epoch, results_epoch_path)
            except Exception as e:
                logging.error(f"Error in config_id={idx}: {str(e)}")
                print(f"Error in config_id={idx}: {str(e)}")
                torch.cuda.empty_cache()
            finally:
                # Clear GPU memory after every configuration
                del pipeline.model
                del pipeline.optimizer
                torch.cuda.empty_cache()

            idx += 1

0.8600064492113302

Running configuration: config_id=2, model_name=indobenchmark/indobert-lite-base-p2, batch_size=4, epochs=100, learning_rate=1e-05


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======


c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\albert\modeling_albert.py:404: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attention_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/100 - Avg training loss: 0.0502, MAE: 0.1834, RMSE: 0.224, Pearson Corr: 0.6884
Avg validation loss: 0.0387, MAE: 0.1631, RMSE: 0.1981, Pearson Corr: 0.7331
Validation loss decreased (inf --> 0.038729). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0339, MAE: 0.1495, RMSE: 0.184, Pearson Corr: 0.8151
Avg validation loss: 0.0356, MAE: 0.1535, RMSE: 0.1899, Pearson Corr: 0.7542
Validation loss decreased (0.038729 --> 0.035570). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0280, MAE: 0.1333, RMSE: 0.1675, Pearson Corr: 0.8452
Avg validation loss: 0.0278, MAE: 0.135, RMSE: 0.168, Pearson Corr: 0.8053
Validation loss decreased (0.035570 --> 0.027836). Saving model ...
====== Training Epoch 4/100 ======
Epoch 4/100 - Avg training loss: 0.0239, MAE: 0.1219, RMSE: 0.1545, Pearson Corr: 0.8699
Avg validation loss: 0.0266, MAE: 0.1281, RMSE: 0.1642, Pearson Corr: 0.7905
Validation loss decreased (0.027836 

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:109: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serializa

Avg testing loss: 0.0158, MAE: 0.09601, RMSE: 0.1261, Pearson Corr: 0.8729
